## Logit Lens on Intervened Prompts

Applies the logit lens to intervened prompts to trace at which layer the model's belief shifts from the factual answer toward the counterfactual one.

In [1]:
%load_ext autoreload
%autoreload 2

### Set-up

In [ ]:
import sys
sys.path.append("src")

import torch
import gc
from tqdm import tqdm

import _config
from _intervention import forward_with_cache, get_logit_lens, get_label_probability_from_logits

In [3]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS_stepwise", # GPT-OSS or R1
    prompt_type="h_pre_result", # empty or pre_result or pre_sum
)
intervention_config = _config.InterventionConfig(
    intervention_loc="final_sum",
    intervention_ids=[25],
)
tok_pos = -1
run_config = _config.RunConfig(
    experiment_root="experiments/token_intervention",
    result_dir="logit_lens",
    output_filename=f"{prompt_config.stem}_{intervention_config.intervention_loc}_{tok_pos}.csv",
)
batch_size = 24

model, tokenizer = _config.load_model(prompt_config.model_type)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

### Model modules (diagnostic)

In [4]:
# Print all modules of the model
print("Model modules:")
for name, module in model.named_modules():
    print(f"{name}: {type(module).__name__}")

Model modules:
: GptOssForCausalLM
model: GptOssModel
model.embed_tokens: Embedding
model.layers: ModuleList
model.layers.0: GptOssDecoderLayer
model.layers.0.self_attn: GptOssAttention
model.layers.0.self_attn.q_proj: Linear
model.layers.0.self_attn.k_proj: Linear
model.layers.0.self_attn.v_proj: Linear
model.layers.0.self_attn.o_proj: Linear
model.layers.0.mlp: GptOssMLP
model.layers.0.mlp.router: GptOssTopKRouter
model.layers.0.mlp.experts: GptOssExperts
model.layers.0.input_layernorm: GptOssRMSNorm
model.layers.0.post_attention_layernorm: GptOssRMSNorm
model.layers.1: GptOssDecoderLayer
model.layers.1.self_attn: GptOssAttention
model.layers.1.self_attn.q_proj: Linear
model.layers.1.self_attn.k_proj: Linear
model.layers.1.self_attn.v_proj: Linear
model.layers.1.self_attn.o_proj: Linear
model.layers.1.mlp: GptOssMLP
model.layers.1.mlp.router: GptOssTopKRouter
model.layers.1.mlp.experts: GptOssExperts
model.layers.1.input_layernorm: GptOssRMSNorm
model.layers.1.post_attention_layernor

In [5]:
prompts = _config.load_prompts(prompt_config)
print(f"loaded {len(prompts)} prompts")

loaded 256 divided prompts


In [6]:
intervention_ids = list(intervention_config.intervention_ids)
print(intervention_ids)

[25]


### Run: per-layer logit lens

Caches each layer's residual-stream activation on the intervened prompt, unembeds it directly, and reads off the factual vs. counterfactual answer probability at every layer.

In [7]:
# Get header of prompts dataset
header = list(prompts.columns) + ['intervention_prompt', 'layer', 'factual_label_probability', 'counterfactual_label_probability']
filepath = _config.build_run_output_filepath(prompt_config, run_config, header)

for i in tqdm(range(0, len(prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = prompts.iloc[i:i+batch_size]

    # Prepare batch of intervention prompts
    intervention_prompts = _config.build_intervened_prompts(batch_rows, intervention_ids)
    tokens = tokenizer(intervention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to('cpu')
    
    base_labels, _ = _config.prepare_sequence_label_tokens(tokenizer, batch_rows['base_sum'], device='cpu')
    source_labels, _ = _config.prepare_sequence_label_tokens(tokenizer, batch_rows['source_sum'], device='cpu')

    output, activations = forward_with_cache(model, tokens['input_ids'], pre_hook=False)
    for layer in range(len(model.model.layers)):
        activation = activations[f'model.layers.{layer}']
        logits = get_logit_lens(model, activation, tok_pos=tok_pos)
        base_labels_probs = get_label_probability_from_logits(logits, base_labels)
        source_labels_probs = get_label_probability_from_logits(logits, source_labels)
    
        # Process each generated text in the batch
        for j, (_, row) in enumerate(batch_rows.iterrows()):
            _config.write_to_csv(filepath, row.to_list() + [intervention_prompts[j], layer, base_labels_probs[j].item(), source_labels_probs[j].item()])


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [01:22<00:00,  7.51s/it]
